In [3]:
#1. VARIEDAD Y FONDO

M = Manifold(4, 'M', latex_name=r'\mathcal{M}')
#Evitamos los simbolos problematicos usando esta forma:

X = M.chart(r't r th:\theta ph:\phi')
t, r, th, ph = X[:]

var('M_m Q pi')
f = 1 - 2*M_m/r + Q^2/r^2

g_bg = M.metric('g_bg')
g_bg[0,0] = -f
g_bg[1,1] = 1/f
g_bg[2,2] = r^2
g_bg[3,3] = r^2 * sin(th)^2
g_inv = g_bg.inverse()
nabla = g_bg.connection()

In [4]:
#Electromagnetismo de Fondo

A_bg = M.one_form('A_bg')
A_bg[0] = -Q/r
F_bg = A_bg.exterior_derivative()
F_bg_up = F_bg.up(g_bg)
F_sq_bg = sum(F_bg[a,b]*F_bg_up[a,b] for a in M.irange() for b in M.irange())

In [5]:
#2. DEFINICIÓN DEL ANSATZ AXIAL

h0 = function('h_0')(t, r)
h1 = function('h_1')(t, r)
a0 = function('a_0')(t, r)
W = function('W')(th)

In [6]:
#Perturbación métrica

h_ax = M.tensor_field(0, 2, 'h_ax', sym=(0,1))
h_ax[0, 3] = h0 * W
h_ax[1, 3] = h1 * W
h_ax_up = h_ax.up(g_bg)

In [7]:
#Perturbación electromagnética

a_ax = M.one_form('a_ax')
a_ax[3] = a0 * W
f_pert = a_ax.exterior_derivative()
f_pert_up = f_pert.up(g_bg)

print("1/4. Fondo y Ansatz configurados...")

1/4. Fondo y Ansatz configurados...


In [10]:
#3. CÁLCULO DE LA CURVATURA

nabla_h = nabla(h_ax)
delta_Gamma = M.tensor_field(1, 2, 'delta_Gamma', sym=(1,2))

for a in M.irange():
    for b in M.irange():
        for c in range(b, 4):
            suma = 0
            for d in M.irange():
                t1 = nabla_h[d, c, b]
                t2 = nabla_h[b, d, c]
                t3 = nabla_h[b, c, d]
                suma += 0.5 * g_inv[a,d] * (t1 + t2 - t3)
                delta_Gamma[a,b,c] = suma

nabla_dGamma = nabla(delta_Gamma)
delta_R = M.tensor_field(0, 2, 'delta_R', sym=(0,1))

for a in M.irange():
    for b in range(a, 4):
        suma1 = sum(nabla_dGamma[c, a, b, c] for c in M.irange())
        suma2 = sum(nabla_dGamma[c, a, c, b] for c in M.irange())
        delta_R[a,b] = suma1 - suma2

print("2/4. Tensor de Ricci linealizado calculado...")

2/4. Tensor de Ricci linealizado calculado...


In [11]:
#4. TENSOR DE ENERGÍA-MOMENTO PERTURBADO

delta_T = M.tensor_field(0, 2, 'delta_T', sym=(0,1))

for mu in M.irange():
    for nu in range(mu, 4):
        t1 = sum(f_pert[mu, alp] * F_bg[nu, bet] * g_inv[alp, bet] for alp in M.irange() for bet in M.irange())
        t2 = sum(F_bg[mu, alp] * f_pert[nu, bet] * g_inv[alp, bet] for alp in M.irange() for bet in M.irange())
        t3 = -sum(F_bg[mu, alp] * F_bg[nu, bet] * h_ax_up[alp, bet] for alp in M.irange() for bet in M.irange())
        t4 = -0.25 * h_ax[mu, nu] * F_sq_bg
        delta_T[mu, nu] = (t1 + t2 + t3 + t4) / (4*pi)

print("3/4. Tensor de Energía-Momento calculado...")

3/4. Tensor de Energía-Momento calculado...


In [12]:
#5. ECUACIÓN DE MAXWELL PERTURBADA

delta_F_up = M.tensor_field(2, 0, 'delta_F_up', antisym=(0,1))

for mu in M.irange():
    for nu in range(mu+1, 4):
        t_f = f_pert_up[mu,nu]
        t_h1 = -sum(h_ax_up[mu, alp] * F_bg[alp, bet] * g_inv[bet, nu] for alp in M.irange() for bet in M.irange())
        t_h2 = -sum(h_ax_up[nu, bet] * F_bg[bet, alp] * g_inv[alp, mu] for alp in M.irange() for bet in M.irange())
        delta_F_up[mu,nu] = t_f + t_h1 + t_h2

nabla_dF_up = nabla(delta_F_up)
delta_M = M.vector_field('delta_M')

for mu in M.irange():
    term1 = sum(nabla_dF_up[mu, nu, nu] for nu in M.irange())
    term2 = sum(delta_Gamma[mu, nu, lam] * F_bg_up[lam, nu] for nu in M.irange() for lam in M.irange())
    delta_M[mu] = term1 + term2

print("4/4. Ecuaciones de Maxwell completadas...\n")

4/4. Ecuaciones de Maxwell completadas...



In [14]:
#6. EXTRACCIÓN DE LAS EXPRESIONES

print("=== LAS 3 ECUACIONES MAESTRAS DEL SISTEMA AXIAL ===")
eq_t_phi = (delta_R[0,3] - 8*pi*delta_T[0,3]).expr()
eq_r_phi = (delta_R[1,3] - 8*pi*delta_T[1,3]).expr()
eq_maxwell = delta_M[3].expr()

print("\n--- 1. Ecuación Einstein (t, phi) ---")
print(eq_t_phi)

print("\n--- 2. Ecuación Einstein (r, phi) ---")
print(eq_r_phi)
print("\n--- 3. Ecuación Maxwell (phi) ---")
print(eq_maxwell)

=== LAS 3 ECUACIONES MAESTRAS DEL SISTEMA AXIAL ===

--- 1. Ecuación Einstein (t, phi) ---
-((-0.5*Q^2*r^2 + 1.0*M_m*r^3 - 0.5*r^4)*cos(th)*h_0(t, r)*diff(W(th), th) + (0.5*Q^2*r^2 - 1.0*M_m*r^3 + 0.5*r^4)*h_0(t, r)*sin(th)*diff(W(th), th, th) + ((-1.0*Q^4 + 4.0*M_m*Q^2*r + 2.0*M_m*r^3 + (-4.0*M_m^2 - 1.0*Q^2)*r^2)*h_0(t, r) + 2*(Q^5 - 4*M_m*Q^3*r - 4*M_m*Q*r^3 + Q*r^4 + 2*(2*M_m^2*Q + Q^3)*r^2)*diff(a_0(t, r), r) + (0.5*Q^4*r^2 - 2.0*M_m*Q^2*r^3 - 2.0*M_m*r^5 + 0.5*r^6 + (2.0*M_m^2 + 1.0*Q^2)*r^4)*diff(h_0(t, r), r, r) + (-1.0*Q^4*r + 4.0*M_m*Q^2*r^2 + 4.0*M_m*r^4 - 1.0*r^5 + (-4.0*M_m^2 - 2.0*Q^2)*r^3)*diff(h_1(t, r), t) + (-0.5*Q^4*r^2 + 2.0*M_m*Q^2*r^3 + 2.0*M_m*r^5 - 0.5*r^6 + (-2.0*M_m^2 - 1.0*Q^2)*r^4)*diff(h_1(t, r), t, r))*W(th)*sin(th))/((Q^2*r^4 - 2*M_m*r^5 + r^6)*sin(th))

--- 2. Ecuación Einstein (r, phi) ---
-((-0.5*Q^2 + 1.0*M_m*r - 0.5*r^2)*cos(th)*h_1(t, r)*diff(W(th), th) + (0.5*Q^2 - 1.0*M_m*r + 0.5*r^2)*h_1(t, r)*sin(th)*diff(W(th), th, th) + (0.5*r^4*diff(h_0(t, r)